# aw_11_seeds — Gate G6: 3-seed final confirmation (protocol §8 + Amendment v1.4)

**Scope (pre-registered in Amendment v1.4):** reseed the *final-stage Phase-2
training* of champion **B4v2** and Track-A control **A2v2** with seeds **43, 44**
(s42 = existing runs-of-record), on identical sha-pinned parents and the identical
frozen data artifact. Full frozen-suite eval per run; greedy decoding.

**Headline requirement:** sign consistency of per-suite B4v2−A2v2 pass-rate deltas
across all 3 seeds; report mean ± sd (single-seed labels removed from final tables
only for the two arms covered here).

Cell order: `a_seeds_setup` → `b_seeds_train`(×4) → `c_seeds_eval`(×4) → `x20_gate`
→ `f_seeds_analysis` → `g_seed_aggregate`.

**Stop rules (§10):** any diverging run is marked `failed` and reported — no
hyperparameter retries. **No new development configs are allowed at this stage.**

**Monitoring per training run (first ~50 steps):** loss finite and decreasing;
eval-JSON validity not collapsing; for SFT, terminal `<|im_end|>` behavior healthy
(v2 adapter contract, modules_to_save=[lm_head, embed_tokens] inherited from the
resolved configs — do NOT re-derive configs by hand).


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt
!python scripts/audit_runtime.py


In [ ]:
# @title a_seeds_setup — runs-of-record + frozen inputs
import subprocess
import json
import os
import pathlib

# ---- runs of record (s42) -------------------------------------------------
B4V2_REPO   = "m97j/aw-runs-b4"          # VERIFY: hub repo holding the B4v2 run
B4V2_RUN    = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"
B4V2_EVAL   = "20260814-032546--eval-playworld--s42--7308ee"
A2V2_REPO   = "REPLACE/aw-runs-a2"       # REPLACE: hub repo holding the A2v2 run
A2V2_RUN    = "REPLACE_A2V2_RUN_ID"      # REPLACE: A2v2 s42 run-of-record id
A2V2_EVAL   = "REPLACE_A2V2_EVAL_RUN_ID" # REPLACE: A2v2 s42 eval run id
P1_PARENT_DIR = None   # filled below from fetched B4v2 lineage
A1V2_PARENT_DIR = None # filled below from fetched A2v2 lineage

SEEDS = [43, 44]  # s42 already exists as run-of-record

def sh(*args):
    print("+", " ".join(args)); r = subprocess.run(list(args))
    assert r.returncode == 0, f"FAILED: {args}"

# Fetch run-of-record dirs (resolved_config.yaml + lineage.json + final_adapter,
# sha-verified by fetch_run). Parents are re-fetched from lineage pins so that
# seed runs share EXACTLY the s42 initialization and data fingerprints.
for repo, run in [(B4V2_REPO, B4V2_RUN), (A2V2_REPO, A2V2_RUN)]:
    sh("python", "scripts/fetch_run.py", "--repo", repo, "--run-id", run)

for run in [B4V2_RUN, A2V2_RUN]:
    lin = json.loads(pathlib.Path(f"runs/{run}/artifacts/lineage.json").read_text())
    print(run, "->", json.dumps(lin, indent=2)[:400])

# Frozen training data: consume ONLY via sha-pinned fetch (Amendment v1.3 rule).
sh("python", "scripts/fetch_dataset.py",
   "--repo", "m97j/aw-playworld", "--path", "train/v1/playworld_sft.jsonl",
   "--expected-sha256", "REPLACE_042eb078_FULL_SHA")
sh("python", "scripts/fetch_dataset.py",
   "--repo", "m97j/aw-playworld", "--path", "train/v1/playworld_preference.jsonl",
   "--expected-sha256", "REPLACE_PREF_FULL_SHA")


In [ ]:
# @title b_seeds_train — B4v2 + A2v2 reseeds (4 runs, sequential)
# Replicate from each run-of-record's resolved_config.yaml, overriding ONLY the
# seed and the experiment name (single-variable discipline). ~4 runs; budget the
# session accordingly (B4v2 SFT ~= s42 wall-clock; A2v2 DPO shorter).
import subprocess

JOBS = []
for seed in SEEDS:
    JOBS.append(dict(
        config=f"runs/{B4V2_RUN}/artifacts/resolved_config.yaml",
        name=f"b4v2-playworld-sft-from-p1-s{seed}",
        parent=f"runs/{B4V2_RUN}/parents/p1_champion/final_adapter",  # from fetch_run lineage materialization; VERIFY path
        repo="m97j/aw-runs-seeds", seed=seed,
    ))
    JOBS.append(dict(
        config=f"runs/{A2V2_RUN}/artifacts/resolved_config.yaml",
        name=f"a2v2-playworld-dpo-s{seed}",
        parent=f"runs/{A2V2_RUN}/parents/a1v2/final_adapter",          # VERIFY path
        repo="m97j/aw-runs-seeds", seed=seed,
    ))

for j in JOBS:
    r = subprocess.run([
        "python", "scripts/run_experiment.py",
        "--config", j["config"],
        "--override", f"runtime.seed={j['seed']}",
        "--override", f"experiment_name={j['name']}",
        "--parent-adapter-dir", j["parent"],
        "--hf-sync-repo", j["repo"],
    ])
    assert r.returncode == 0, f"train failed: {j['name']} — STOP, do not retune (protocol §10)"


In [ ]:
# @title c_seeds_eval — frozen-suite eval for all 4 new adapters
import subprocess
import glob
import os

ADAPTERS = {}
for j in JOBS:
    cand = sorted(glob.glob(f"runs/*--{j['name']}--s{j['seed']}--*/artifacts/final_adapter"))
    assert cand, f"no adapter for {j['name']}"
    ADAPTERS[(j['name'], j['seed'])] = cand[-1]

EVAL_RUNS = {}
for key, adapter in ADAPTERS.items():
    r = subprocess.run([
        "python", "scripts/run_evaluation.py",
        "--config", "configs/experiments/eval_playworld.yaml",
        "--adapter-dir", adapter,
        "--hf-sync-repo", "m97j/aw-runs-seeds",
    ])
    assert r.returncode == 0, f"eval failed: {key}"
    EVAL_RUNS[key] = sorted(glob.glob("runs/*--eval-playworld--*"))[-1]
print(EVAL_RUNS)


In [ ]:
# @title x20_gate — eval identity audit (mandatory after the stale-weights incident)
# Every pair of eval runs that should differ MUST NOT be prediction-identical.
import subprocess
import itertools
runs = list(EVAL_RUNS.values()) + [f"runs/{B4V2_EVAL}"]
for a, b in itertools.combinations(runs, 2):
    r = subprocess.run(["python", "scripts/x20_eval_identity_audit.py",
                        "--run-a", a, "--run-b", b])
    assert r.returncode == 0, f"identity audit FAILED (stale weights?): {a} vs {b}"


In [ ]:
# @title f_seeds_analysis — per-seed paired B4v2 vs A2v2 + champion stability
import subprocess
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    b = EVAL_RUNS[(f"a2v2-playworld-dpo-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", b, "--label-b", f"a2v2-s{seed}",
        "--output", f"{a}/analysis_b4v2_vs_a2v2_s{seed}.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)
# seed-to-seed drift of the champion itself (s43/s44 vs s42 run-of-record):
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", f"runs/{B4V2_EVAL}", "--label-b", "b4v2-s42",
        "--output", f"{a}/analysis_b4v2_s{seed}_vs_s42.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)


In [ ]:
# @title g_seed_aggregate — x21 mean±sd + sign-consistency verdict (G6 gate)
import subprocess
args = ["python", "scripts/x21_seed_variance.py", "--output", "runs/seed_variance_report.json"]
args += ["--model", "b4v2", "--eval-run", f"runs/{B4V2_EVAL}", "--seed", "42"]
args += ["--model", "a2v2", "--eval-run", f"runs/{A2V2_EVAL}", "--seed", "42"]
for (name, seed), run in EVAL_RUNS.items():
    model = "b4v2" if name.startswith("b4v2") else "a2v2"
    args += ["--model", model, "--eval-run", run, "--seed", str(seed)]
subprocess.run(args, check=True)
subprocess.run(["python", "-c",
  "import json;print(json.dumps(json.load(open('runs/seed_variance_report.json'))['verdict'],indent=2))"],
  check=True)


## Deliverables to bring back to the assistant after this notebook
1. `runs/seed_variance_report.json` (full JSON)
2. The four `analysis_*.json` outputs of `f_seeds_analysis`
3. run_ids + adapter sha256 of the 4 new runs (from run_card.json)
4. Any `failed` run's event log tail if a stop rule fired
5. x20 gate output (PASS lines)

If the sign-consistency verdict is PASS → proceed to §7 write-up (tech report).
If any suite flips sign across seeds → the report's headline is weakened to the
suites that remain consistent; do NOT rerun with new seeds (that would be
seed-shopping and violates §10).
